# H0 + Train-only Prototype Similarity Screen

## Goal
H0 Selective-EB가 전역 암종 증거를 합산하는 방식과 별개로, fold-train 환자들의 mutation profile class prototype과의 유사도를 사용한다.

- Screen: seed 42, Stratified 5-fold
- Variants: H0 / prototype-only / fixed `0.80 H0 + 0.20 prototype`
- Promotion: H0 대비 Macro F1 `+0.015` 이상, 4/5 fold 상승, low-margin 붕괴 없음

## Safety contract
- `train.csv`만 읽는다. test 통계·vocabulary·scaling은 사용하지 않는다.
- event vocabulary, IDF, priors, prototypes는 outer-fold train에서만 fit한다.
- WT·빈 문자열·NaN은 event가 아니며 `nan_as_mutation_count == 0`을 확인한다.
- 고정 암종명·유전자명·mutation 목록은 사용하지 않는다.


## 1. Setup
이 노트북은 실행기를 호출하는 reader-facing companion입니다. 실행 중에는 fold checkpoint가 저장되므로 중단 후 재실행할 수 있습니다.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

ROOT = Path('/Users/admin/Documents/FinalProject/OZ_fianl_hackaton')
RUNNER = ROOT / 'experiments/gs/notebooks/exp_model_012/common/run_h0_prototype_similarity_screen.py'
RESULT = ROOT / 'experiments/gs/notebooks/exp_model_012/result'
RUN_ID = 'exp-h0-prototype-similarity-01'
RUN_EXPERIMENT = True

assert RUNNER.exists(), RUNNER
print({'runner': RUNNER, 'result_dir': RESULT, 'run_experiment': RUN_EXPERIMENT})


## 2. Run seed42 screen
전체 CV는 이 셀에서만 실행합니다. `RUN_EXPERIMENT=False`로 바꾸면 기존 결과만 읽습니다.


In [ ]:
if RUN_EXPERIMENT:
    process = subprocess.Popen(
        [sys.executable, str(RUNNER), '--run-id', RUN_ID],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail = []
    for line in tqdm(process.stdout, desc='prototype screen', unit='line'):
        print(line, end='')
        tail = (tail + [line])[-120:]
    if process.wait():
        raise RuntimeError('prototype screen failed:\n' + ''.join(tail))
else:
    print('RUN_EXPERIMENT=False: existing result files only')


## 3. Results and decision
동일 outer-fold validation에서 H0 대비 paired 결과를 확인합니다.


In [ ]:
prefix = RESULT / f'{RUN_ID}_seed42'
summary = pd.read_csv(prefix.with_name(prefix.name + '_summary.csv'))
folds = pd.read_csv(prefix.with_name(prefix.name + '_fold_metrics.csv'))
classes = pd.read_csv(prefix.with_name(prefix.name + '_class_metrics.csv'))
low_margin = pd.read_csv(prefix.with_name(prefix.name + '_low_margin.csv'))
audit = json.loads(prefix.with_name(prefix.name + '_leakage_audit.json').read_text())

assert summary.leakage_check.all()
assert summary.nan_as_mutation_count.eq(0).all()
display(summary.sort_values('oof_macro_f1', ascending=False))
display(folds.pivot(index='fold', columns='variant', values='macro_f1'))
display(low_margin)
audit


In [ ]:
folds.pivot(index='fold', columns='variant', values='macro_f1').plot(marker='o', figsize=(8, 4), title='H0 vs prototype similarity')
plt.ylabel('Macro F1')
plt.tight_layout()
plt.show()

h0 = classes[classes.variant.eq('H0_selective_EB')].set_index('class').f1
blend = classes[classes.variant.eq('H0_plus_prototype')].set_index('class').f1
(blend - h0).sort_values().plot.barh(figsize=(7, 7), title='Prototype blend: class F1 delta')
plt.axvline(0, color='black')
plt.tight_layout()
plt.show()

print('Automatic decision:', '3-seed candidate' if audit['screen_candidate'] else 'screen rejected')
print(audit)
